# Closed-form $f$ by direct distillation, with the CBE as the verdict

## Why the joint searches failed, and why this one cannot

Four joint searches for $(f, \Phi)$ all returned the same degenerate answer:
delete $z$ from $\Phi$, make $g$ trivial. That was not bad luck — it was the
optimum of the objective I wrote. **The CBE alone contains no information about
$\Phi$**: a uniform $f$ is constant along every orbit in every potential, so
$f = {\rm const}$ solves it exactly for any $\Phi$ whatsoever. Making the
residual the thing to minimise invites exactly that.

The information about $\Phi$ lives in how the velocity distribution *changes
with position*:

| $\vert z\vert$ (kpc) | $\sigma_R$ | $\sigma_z$ |
|---|---|---|
| < 0.1 | 35.2 | 18.2 |
| 0.2–0.4 | 40.5 | 22.1 |
| 0.4–0.7 | 44.8 | 25.6 |

So this notebook **stops optimising the residual**. Instead:

1. $g = \ln f$ is fitted by **supervised regression** to the empirical
   $\ln \hat f$ from notebook 2 — a target that varies strongly (sd 1.94) and
   is already known at every star. A constant $g$ scores catastrophically
   against it, so the degenerate solution is simply unavailable.
2. The CBE residual is then **computed, not minimised** — a verdict on the
   result rather than a knob the search can game.

This is the identical two-stage structure that already worked for $\Phi$ in
notebook 3 (measure the field, then fit it), now applied to $f$. And it is
plain regression against a smooth target, which is what symbolic regression is
actually good at — not a coupled evolutionary search over seven variables.

**Sobolev fitting.** The CBE consumes $\nabla \ln f$, not $\ln f$, so matching
values alone would let the derivatives drift. All six gradient targets are
already in `df_gradients.parquet` (verified against finite differences to
$10^{-6}$ in notebook 2), so the loss matches value **and** gradients.

**What the empirical target already accounts for**, so this inherits it:
the selection function (via the conditional velocity distribution, which is
selection-free exactly, plus the fitted disc density), and the azimuthal noise
(dropped — $\partial_\phi \ln f$ was 93% of the streaming term and
irreproducible between refits).

In [1]:
from pathlib import Path
import json
import os
import sys
import time

os.environ.setdefault("PYTHON_JULIACALL_THREADS", "auto")
os.environ.setdefault("PYTHON_JULIACALL_AUTOLOAD_IPYTHON_EXTENSION", "no")

import numpy as np
import pandas as pd
import sympy

sys.path.insert(0, str(Path.cwd()))
from gaia2 import (LOGCOSH_JULIA, LOGG_JULIA, SQRTG_JULIA, cbe_terms,
                   density_from_laplacian, split_template_equation,
                   streaming_term, surface_density_from_kz, sympify_expression,
                   sympy_operator_map)
from pysr import PySRRegressor, TemplateExpressionSpec

HERE = Path("/ocean/projects/cis240096p/idas/iaifi_summer_school/gaia_project/notebooks/final_claude")
TRIAL2 = HERE.parent / "trial_2_claude"
OUT = HERE / "outputs"
OUT.mkdir(exist_ok=True)

frame = json.loads((TRIAL2 / "data/frame.json").read_text())
R0, SEED = frame["R0"], frame["seed"]
potential = json.loads((TRIAL2 / "outputs/03_potential_result.json").read_text())
U = 100.0
INNER_CUT = 0.9

work = pd.read_parquet(TRIAL2 / "data/df_gradients.parquet")
work = work.loc[work["r_helio"] < INNER_CUT].reset_index(drop=True)
print(f"{len(work):,} stars with empirical ln f and its six gradients")
print(f"target ln f: sd = {work['log_f'].std():.3f}  (a constant g scores exactly this)")
print(f"notebook 3 potential: {potential['chosen_equation']}")

2,214,820 stars with empirical ln f and its six gradients
target ln f: sd = 1.927  (a constant g scores exactly this)
notebook 3 potential: Phi = logg(#1 + (logcosh(#2) * 0.39193293)) * 5.361797


## 1. Targets

Velocities in units of 100 km/s, so the velocity gradients are multiplied by
100 to match. Every target is normalised by its own rms so that no single
component dominates the loss.

In [2]:
N_FIT = 20_000
GRAD = ["g_R", "g_z", "g_vR", "g_vphi", "g_vz"]

fit = work.loc[work["split"] == "train"].sample(n=N_FIT, random_state=SEED)
offset = float(fit["log_f"].mean())


def build(subset):
    out = pd.DataFrame({
        "R": subset["R"].to_numpy(), "z": subset["zc"].to_numpy(),
        "vR": subset["vR"].to_numpy() / U, "vphi": subset["vphi"].to_numpy() / U,
        "vz": subset["vzc"].to_numpy() / U,
        "tg": subset["log_f"].to_numpy() - offset,
        "t1": subset["g_R"].to_numpy(), "t2": subset["g_z"].to_numpy(),
        "t3": subset["g_vR"].to_numpy() * U, "t4": subset["g_vphi"].to_numpy() * U,
        "t5": subset["g_vz"].to_numpy() * U})
    return out


rows = build(fit)
COLUMNS = ["R", "z", "vR", "vphi", "vz", "tg", "t1", "t2", "t3", "t4", "t5"]
X = rows[COLUMNS].to_numpy(np.float64)
y = np.zeros(len(X))

SG = float(np.sqrt(np.mean(rows["tg"] ** 2)))
S1, S2 = float(np.sqrt(np.mean(rows["t1"] ** 2))), float(np.sqrt(np.mean(rows["t2"] ** 2)))
S3 = float(np.sqrt(np.mean(rows["t3"] ** 2)))
S4, S5 = float(np.sqrt(np.mean(rows["t4"] ** 2))), float(np.sqrt(np.mean(rows["t5"] ** 2)))
print(f"rms targets: value {SG:.3f} | dR {S1:.3f} dz {S2:.3f} "
      f"| dvR {S3:.3f} dvphi {S4:.3f} dvz {S5:.3f}")

GRAD_WEIGHT = 1.0   # value and gradients weighted equally after normalising

rms targets: value 1.931 | dR 1.725 dz 3.060 | dvR 3.413 dvphi 6.725 dvz 6.451


## 2. The search

One expression. A constant `g` gives loss 1.0 on the value term alone and 1.0
on every gradient term — there is nowhere to hide.

In [3]:
A = "R,z,vR,vphi,vz"
COMBINE = (
    "d1 = D(g,1); d2 = D(g,2); d3 = D(g,3); d4 = D(g,4); d5 = D(g,5); "
    f"e0 = (g({A}) - tg)/{SG}f0; "
    f"e1 = (d1({A}) - t1)/{S1}f0; e2 = (d2({A}) - t2)/{S2}f0; "
    f"e3 = (d3({A}) - t3)/{S3}f0; e4 = (d4({A}) - t4)/{S4}f0; "
    f"e5 = (d5({A}) - t5)/{S5}f0; "
    f"sqrt(e0*e0 + {GRAD_WEIGHT}f0*(e1*e1 + e2*e2 + e3*e3 + e4*e4 + e5*e5)/5.0f0)"
)
print(COMBINE.replace("; ", ";\n"))

search = PySRRegressor(
    expression_spec=TemplateExpressionSpec(
        expressions=["g"], variable_names=COLUMNS, combine=COMBINE),
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["exp", LOGCOSH_JULIA, LOGG_JULIA, SQRTG_JULIA],
    extra_sympy_mappings=sympy_operator_map(),
    elementwise_loss="(p, t) -> p^2",
    niterations=1000, populations=60, population_size=40,
    ncycles_per_iteration=400, batching=True, batch_size=2000,
    maxsize=45, maxdepth=14, timeout_in_seconds=4200,
    complexity_of_constants=2, parsimony=1e-3,
    nested_constraints={"exp": {"exp": 0}, "logg": {"logg": 0}},
    parallelism="multithreading", random_state=SEED,
    progress=False, verbosity=0,
    output_directory=str(OUT / "pysr_f"))

t0 = time.time()
search.fit(X, y, variable_names=COLUMNS)
print(f"search finished in {time.time() - t0:.0f}s")
search.equations_[["complexity", "loss", "score"]]

d1 = D(g,1);
d2 = D(g,2);
d3 = D(g,3);
d4 = D(g,4);
d5 = D(g,5);
e0 = (g(R,z,vR,vphi,vz) - tg)/1.9307901037616957f0;
e1 = (d1(R,z,vR,vphi,vz) - t1)/1.7248062158083552f0;
e2 = (d2(R,z,vR,vphi,vz) - t2)/3.059635072153162f0;
e3 = (d3(R,z,vR,vphi,vz) - t3)/3.413116860389776f0;
e4 = (d4(R,z,vR,vphi,vz) - t4)/6.72470519174101f0;
e5 = (d5(R,z,vR,vphi,vz) - t5)/6.45122799910616f0;
sqrt(e0*e0 + 1.0f0*(e1*e1 + e2*e2 + e3*e3 + e4*e4 + e5*e5)/5.0f0)


/ocean/projects/cis240096p/idas/iaifi_summer_school/.venv/lib/python3.12/site-packages/pysr/sr.py:1469: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
/ocean/projects/cis240096p/idas/iaifi_summer_school/.venv/lib/python3.12/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


search finished in 4809s


,complexity,loss,score
0,1,2.016289,0.000000
1,2,2.000000,0.008112
2,3,1.941455,0.029709
3,4,1.847667,0.049514
4,5,1.578841,0.157232
5,6,1.566474,0.007864
6,7,1.408909,0.106011
7,8,1.372857,0.025922
8,9,1.352369,0.015036
9,10,1.312633,0.029823


## 3. Score every candidate on held-out stars

Three independent things, none of which the search optimised: how well $g$
reproduces the empirical $\ln f$, the velocity ellipsoid it implies, and the
CBE residual against notebook 3's $\Phi$.

In [4]:
SYM = {n: sympy.Symbol(n, real=True) for n in ["R", "z", "vR", "vphi", "vz"]}
ARGS = tuple(SYM[n] for n in ["R", "z", "vR", "vphi", "vz"])
Rs, zs = SYM["R"], SYM["z"]

phi_pieces, _ = split_template_equation(potential["chosen_equation"])
PHI = sympify_expression(phi_pieces["Phi"], ["R", "z"], SYM)
dPR = sympy.lambdify((Rs, zs), sympy.diff(PHI, Rs), "numpy")
dPz = sympy.lambdify((Rs, zs), sympy.diff(PHI, zs), "numpy")

test = work.loc[work["split"] == "test"]
tr = build(test)
Rt, zt = tr["R"].to_numpy(), tr["z"].to_numpy()
vRt, vpt, vzt = tr["vR"].to_numpy(), tr["vphi"].to_numpy(), tr["vz"].to_numpy()
targets = [tr[c].to_numpy() for c in ["tg", "t1", "t2", "t3", "t4", "t5"]]

known_t, weights_t, _ = cbe_terms(
    test["R"].to_numpy(), test["vR"].to_numpy(), test["vphi"].to_numpy(),
    test["vzc"].to_numpy(), test[["g_R", "g_phi", "g_z", "g_vR", "g_vphi", "g_vz"]].to_numpy(),
    include_phi=False)
stream_t = streaming_term(
    test["R"].to_numpy(), test["vR"].to_numpy(), test["vphi"].to_numpy(),
    test["vzc"].to_numpy(), test[["g_R", "g_phi", "g_z", "g_vR", "g_vphi", "g_vz"]].to_numpy(),
    include_phi=False)
EMP_CBE = float(np.sqrt(np.mean(
    (known_t + weights_t @ np.array([-dPR(R0, 0.0) * U * U, 0.0, 0.0])) ** 2)))
print(f"empirical f with this Phi: reference residual scale {np.sqrt(np.mean(known_t**2)):.1f}")


def bc(fn, *a):
    with np.errstate(all="ignore"):
        return np.broadcast_to(np.asarray(fn(*a), dtype=np.float64), np.shape(a[0])).copy()


def assess(text):
    pieces, _ = split_template_equation(text)
    g = sympify_expression(pieces["g"], ["R", "z", "vR", "vphi", "vz"], SYM)
    gv = bc(sympy.lambdify(ARGS, g, "numpy"), Rt, zt, vRt, vpt, vzt)
    d = [bc(sympy.lambdify(ARGS, sympy.diff(g, s), "numpy"), Rt, zt, vRt, vpt, vzt)
         for s in ARGS]
    ok = np.isfinite(gv) & np.all(np.isfinite(d), axis=0)
    if ok.sum() < 1000:
        return None
    # how well does it reproduce the empirical ln f and its gradients?
    r2 = 1.0 - np.mean((gv[ok] - targets[0][ok]) ** 2) / np.var(targets[0][ok])
    grad_rel = float(np.mean([np.sqrt(np.mean((d[i][ok] - targets[i + 1][ok]) ** 2)
                              / np.mean(targets[i + 1][ok] ** 2)) for i in range(5)]))
    # CBE with notebook 3's Phi, using this g's OWN exact derivatives
    aR = -bc(dPR, Rt, zt) * U * U
    az = -bc(dPz, Rt, zt) * U * U
    Rk = test["R"].to_numpy()[ok]
    vRk, vpk, vzk = (test[c].to_numpy()[ok] for c in ["vR", "vphi", "vzc"])
    gR, gz = d[0][ok], d[1][ok]
    gvR, gvp, gvz = d[2][ok] / U, d[3][ok] / U, d[4][ok] / U
    known = vRk * gR + vzk * gz + (vpk * vpk / Rk) * gvR - (vRk * vpk / Rk) * gvp
    cbe = known + aR[ok] * gvR + az[ok] * gvz
    return {"lnf_R2": float(r2), "grad_rel_err": grad_rel,
            "cbe_kms_kpc": float(np.sqrt(np.mean(cbe ** 2))),
            "cbe_over_known": float(np.sqrt(np.mean(cbe ** 2) / np.mean(known ** 2))),
            "finite": float(ok.mean())}


def ellipsoid(text, n=80_000, seed=0):
    pieces, _ = split_template_equation(text)
    g = sympify_expression(pieces["g"], ["R", "z", "vR", "vphi", "vz"], SYM)
    fn = sympy.lambdify(ARGS, g, "numpy")
    rng = np.random.default_rng(seed)
    c = np.array([0.0, -222.0, 0.0]) / U
    s = np.array([70.0, 62.0, 50.0]) / U
    u = c + rng.normal(size=(n, 3)) * s
    lq = -0.5 * np.sum(((u - c) / s) ** 2, axis=1)
    lg = bc(fn, np.full(n, R0), np.zeros(n), u[:, 0], u[:, 1], u[:, 2])
    m = np.isfinite(lg)
    if m.sum() < 1000:
        return np.nan
    w = np.zeros(n)
    sh = lg[m] - lq[m]
    w[m] = np.exp(np.clip(sh - sh.max(), -700, 0))
    if w.sum() <= 0:
        return np.nan
    w /= w.sum()
    cen = u - (w @ u)
    return float(np.sqrt(w @ cen[:, 0] ** 2) / np.sqrt(w @ cen[:, 2] ** 2))


records = []
for _, row in search.equations_.iterrows():
    entry = {"complexity": int(row["complexity"]), "loss": float(row["loss"]),
             "equation": str(row["equation"])}
    try:
        entry.update(assess(str(row["equation"])) or {})
        entry["sigma_R_over_z"] = ellipsoid(str(row["equation"]))
    except Exception:
        pass
    records.append(entry)
front = pd.DataFrame(records)
front.to_csv(OUT / "f_front.csv", index=False)

near = work.loc[(np.abs(work["R"] - R0) < 0.15) & (np.abs(work["zc"]) < 0.1)]
obs_ratio = float(near["vR"].std() / near["vzc"].std())
print(f"observed sigma_R/sigma_z = {obs_ratio:.2f}")
print(f"empirical f, same Phi, held out: 58.6 (km/s)/kpc with a 48.7 noise floor\n")
front[[c for c in ["complexity", "loss", "lnf_R2", "grad_rel_err",
                   "cbe_kms_kpc", "cbe_over_known", "sigma_R_over_z"] if c in front]]

empirical f with this Phi: reference residual scale 232.5


/var/tmp/ipykernel_1947/3282232357.py:58: RuntimeWarning: divide by zero encountered in scalar divide
  "cbe_over_known": float(np.sqrt(np.mean(cbe ** 2) / np.mean(known ** 2))),


/var/tmp/ipykernel_1947/3282232357.py:58: RuntimeWarning: invalid value encountered in scalar divide
  "cbe_over_known": float(np.sqrt(np.mean(cbe ** 2) / np.mean(known ** 2))),


observed sigma_R/sigma_z = 1.89
empirical f, same Phi, held out: 58.6 (km/s)/kpc with a 48.7 noise floor



,complexity,loss,lnf_R2,grad_rel_err,cbe_kms_kpc,cbe_over_known,sigma_R_over_z
0,1,2.016289,-7.632839e-03,1.002357,6.157640e+00,inf,1.137161
1,2,2.000000,-2.746953e-05,1.000000,0.000000e+00,NaN,0.911633
2,3,1.941455,6.248420e-02,1.026883,6.987356e+00,1.000000,0.881938
3,4,1.847667,9.130061e-02,0.967507,1.222115e+01,0.267465,0.774935
4,5,1.578841,3.085066e-01,0.931327,2.981029e+01,inf,9.453903
5,6,1.566474,2.857588e-01,0.901036,3.966506e+01,3.165841,10.854256
6,7,1.408909,4.469545e-01,0.897407,3.912344e+01,3.122612,11.206332
7,8,1.372857,4.800972e-01,0.924290,4.086522e+01,2.302133,9.052189
8,9,1.352369,5.098840e-01,0.899673,4.167500e+01,2.108429,8.108547
9,10,1.312633,-2.477981e+09,34144.796055,1.355814e+08,1.151938,9.897336


## 4. The answer

In [5]:
ok = front.dropna(subset=["lnf_R2"])
good = ok.loc[(ok["finite"] > 0.99) & (ok["lnf_R2"] > 0.5)]
note = "reproduces the empirical ln f with R^2 > 0.5"
if len(good) == 0:
    good, note = ok, "WARNING: nothing reached R^2 > 0.5 against the empirical ln f"
best = good.sort_values("lnf_R2", ascending=False).iloc[0]

pieces, _ = split_template_equation(str(best["equation"]))
g_expr = sympify_expression(pieces["g"], ["R", "z", "vR", "vphi", "vz"], SYM)
print(f"selection: {note}\n")
print("f = exp(g),  with")
print("  g   =", sympy.simplify(g_expr))
print("  Phi =", sympy.simplify(PHI), "   [from notebook 3]")
print(f"\n  complexity                 {best['complexity']}")
print(f"  R^2 against empirical ln f {best['lnf_R2']:.4f}")
print(f"  mean relative gradient err {best['grad_rel_err']:.4f}")
print(f"  CBE residual               {best['cbe_kms_kpc']:.1f} (km/s)/kpc"
      f"   = {best['cbe_over_known']:.3f} of its known part")
print(f"  sigma_R/sigma_z            {best['sigma_R_over_z']:.2f}   (observed {obs_ratio:.2f})")

result = {
    "note": note, "g": str(g_expr), "Phi": str(PHI),
    "complexity": int(best["complexity"]),
    "lnf_R2": float(best["lnf_R2"]), "grad_rel_err": float(best["grad_rel_err"]),
    "cbe_kms_kpc": float(best["cbe_kms_kpc"]),
    "cbe_over_known": float(best["cbe_over_known"]),
    "sigma_ratio_model": float(best["sigma_R_over_z"]),
    "sigma_ratio_observed": obs_ratio,
    "rho_midplane": potential["rho_midplane"], "v_c": potential["v_c"],
    "Sigma_half_kpc": potential["Sigma_half_kpc"],
    "reference": {"empirical f, same Phi": 58.6, "estimator noise floor": 48.7,
                  "trial 1 relative residual": 0.958},
}
(OUT / "final_result.json").write_text(json.dumps(result, indent=2, default=float))
print("\n" + json.dumps(result, indent=2, default=float))

selection: reproduces the empirical ln f with R^2 > 0.5

f = exp(g),  with


  g   = -log(cosh(8.52310161122417*vz)*cosh((log(cosh(vphi)) - 1.6250938)*(-R + log(cosh(5.2512811550634*vz)) + 1.3851725))*cosh(log(cosh(2.10886732314301*vR*vphi)))) - log(cosh(z*log(cosh(vphi*log(R))))) + 2.4066026
  Phi = 5.361797*log(R + 0.39193293*log(cosh(z)))    [from notebook 3]

  complexity                 44
  R^2 against empirical ln f 0.9228
  mean relative gradient err 0.6950
  CBE residual               38.2 (km/s)/kpc   = 0.204 of its known part
  sigma_R/sigma_z            1.88   (observed 1.89)

{
  "note": "reproduces the empirical ln f with R^2 > 0.5",
  "g": "-log(cosh(8.52310161122417*vz)) - log(cosh(z*log(cosh(vphi*log(R))))) - log(cosh((log(cosh(vphi)) - 1.6250938)*(-R + log(cosh(5.2512811550634*vz)) + 1.3851725))) - log(cosh(log(cosh(2.10886732314301*vR*vphi)))) + 2.4066026",
  "Phi": "5.361797*log(R + 0.39193293*log(cosh(z)))",
  "complexity": 44,
  "lnf_R2": 0.9227771143830229,
  "grad_rel_err": 0.694969471199934,
  "cbe_kms_kpc": 38.15075082567336,
  "cbe_ov